# XQuality Prefix Stop-After-Layer Evaluation — fixed exact NeoOLAF evaluator

This notebook evaluates already generated `prefix_stop_after_*` outputs using the **same NeoOLAF extraction evaluator** as the original ablation pipeline.

It fixes the failure mode where every metric becomes zero because the gold file was loaded with zero gold entities/relations.

What this notebook does:

1. Finds the NeoOLAF project root.
2. Finds and validates the correct XQuality gold file.
3. Refuses to continue if gold relations/entities are empty.
4. Evaluates every completed `prefix_stop_after_*` output folder.
5. Adds native NeoOLAF references:
   - native Layer 12 state via `evaluate_layer_state`;
   - native final export JSON, when available.
6. Saves summary CSVs and plots.

It does **not** call OpenRouter. It only evaluates saved outputs.

In [ ]:
from __future__ import annotations

import json
import re
import sys
import math
import shutil
from dataclasses import asdict
from pathlib import Path
from collections import Counter
from typing import Any

import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)


## 1. Locate project root and configure paths

In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    candidates = [start] + list(start.parents)
    for p in candidates:
        if (p / "src" / "neoolaf").exists() and (p / "examples" / "XQualityMachine32").exists():
            return p
    # fallback: search common Windows path when notebook is inside repo but cwd differs
    for p in [Path.cwd().resolve(), Path.cwd().resolve().parent]:
        if (p / "src" / "neoolaf").exists():
            return p
    raise RuntimeError("Could not find NeoOLAF project root. Run this notebook from inside the NeoOLAF repo.")

ROOT = find_project_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT =", ROOT)
print("SRC  =", SRC)

# Same profile you used in the original ablation / native final reference.
PROFILE_NAME = "xquality_loose"

# Folder produced by the prefix stop-after-layer generation notebook.
PREFIX_OUTPUT_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "prefix_stop_after_layer_llm_finalization_exact_eval"

# Where this fixed evaluation will be saved.
EVAL_OUTPUT_DIR = ROOT / "examples" / "XQualityMachine32" / "runs" / "prefix_stop_after_layer_exact_eval_fixed"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PREFIX_OUTPUT_DIR =", PREFIX_OUTPUT_DIR, "exists=", PREFIX_OUTPUT_DIR.exists())
print("EVAL_OUTPUT_DIR   =", EVAL_OUTPUT_DIR)


## 2. Import exact NeoOLAF evaluation functions

In [ ]:
from neoolaf.evaluation.runners.evaluate_layer_state import (
    evaluate_layer_state,
    load_xquality_gold_any,
    gold_to_artifact,
)
from neoolaf.evaluation.datasets.xquality import load_xquality_gold
from neoolaf.evaluation.metrics.extraction import evaluate_extraction
from neoolaf.evaluation.profiles.registry import get_profile
from neoolaf.evaluation.schema.artifact import (
    EvaluationArtifact,
    EvalDocument,
    EvalEntity,
    EvalRelation,
)

profile = get_profile(PROFILE_NAME)
print("Loaded profile:", profile.name)


## 3. Find and validate the correct gold file

If the gold file is loaded incorrectly, the evaluator can return `fn = 0` and every metric becomes zero. This cell prevents that by requiring non-empty gold entities and relations.

In [ ]:
GOLD_CANDIDATES = [
    ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json",
    ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.json",
    ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xlsx",
    ROOT / "data" / "XQuality" / "Machine32" / "machine32_triples.xls",
]

# Add recursive candidates as a fallback, but keep explicit ones first.
for p in (ROOT / "data").rglob("*.json"):
    name = p.name.lower()
    if "trip" in name or "gold" in name or "xquality" in name:
        if p not in GOLD_CANDIDATES:
            GOLD_CANDIDATES.append(p)
for p in (ROOT / "data").rglob("*.xlsx"):
    name = p.name.lower()
    if "trip" in name or "gold" in name or "machine32" in name:
        if p not in GOLD_CANDIDATES:
            GOLD_CANDIDATES.append(p)


def try_load_gold(path: Path):
    try:
        gold = load_xquality_gold_any(path)
        gold_artifact = gold_to_artifact(gold, profile=PROFILE_NAME)
        n_entities = sum(len(v) for v in gold_artifact.entities_by_doc.values())
        n_relations = sum(len(v) for v in gold_artifact.relations_by_doc.values())
        return gold, gold_artifact, n_entities, n_relations, None
    except Exception as e:
        return None, None, 0, 0, repr(e)

rows = []
for path in GOLD_CANDIDATES:
    if not path.exists():
        rows.append({"path": str(path), "exists": False, "gold_entities": 0, "gold_relations": 0, "error": "missing"})
        continue
    gold, gold_artifact, ne, nr, err = try_load_gold(path)
    rows.append({"path": str(path), "exists": True, "gold_entities": ne, "gold_relations": nr, "error": err})

gold_probe_df = pd.DataFrame(rows).drop_duplicates(subset=["path"])
display(gold_probe_df.sort_values(["gold_relations", "gold_entities"], ascending=False).head(20))

valid_gold_rows = gold_probe_df[(gold_probe_df["exists"] == True) & (gold_probe_df["gold_relations"] > 0) & (gold_probe_df["gold_entities"] > 0)].copy()
if valid_gold_rows.empty:
    raise RuntimeError(
        "No usable XQuality gold file was found. This would make all metrics zero. "
        "Check that the gold file has columns 'Node 1', 'RELATION', 'Node 2'."
    )

# Prefer the known final-evaluation file when valid; otherwise choose the candidate with the most relations.
preferred = ROOT / "data" / "XQuality" / "Examples" / "XQuality_all_triplets_flat_en.json"
if str(preferred) in set(valid_gold_rows["path"]):
    GOLD_PATH = preferred
else:
    GOLD_PATH = Path(valid_gold_rows.sort_values(["gold_relations", "gold_entities"], ascending=False).iloc[0]["path"])

gold, gold_artifact, gold_entity_count, gold_relation_count, err = try_load_gold(GOLD_PATH)
print("Selected GOLD_PATH:", GOLD_PATH)
print("Gold entities:", gold_entity_count)
print("Gold relations:", gold_relation_count)
print("Gold relation labels:", Counter(r.relation for rs in gold_artifact.relations_by_doc.values() for r in rs))

# Hard sanity check: your final gold evaluation had non-zero FN, so gold must not be empty.
assert gold_entity_count > 0, "Gold entity count is zero. Evaluation would be invalid."
assert gold_relation_count > 0, "Gold relation count is zero. Evaluation would be invalid."


## 4. Robust triple extraction from generated outputs and final exports

In [ ]:
RELATION_ALIASES = {
    "TRIGGER": "TRIGGERS",
    "TRIGGERS": "TRIGGERS",
    "CAUSE": "CAUSES",
    "CAUSES": "CAUSES",
    "EFFECT": "CAUSES",
    "REQUIRES": "REQUIRES",
    "REQUIRE": "REQUIRES",
    "REQUIRED_ACTION": "REQUIRES",
    "INTERVENTION": "REQUIRES",
    "ACTION": "REQUIRES",
    "HANDLED_BY": "HANDLED_BY",
    "HANDLED BY": "HANDLED_BY",
    "RESPONSIBLE": "HANDLED_BY",
    "RESPONSABLE": "HANDLED_BY",
    "REFERENCES": "REFERENCES",
    "REFERENCE": "REFERENCES",
    "REFERS_TO": "REFERENCES",
}


def clean_text(x: Any) -> str:
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    if isinstance(x, dict):
        for k in ["label", "name", "text", "value", "canonical_label", "id"]:
            if k in x and clean_text(x[k]):
                return clean_text(x[k])
        return ""
    return re.sub(r"\s+", " ", str(x).strip())


def normalize_predicate(p: Any) -> str:
    s = clean_text(p)
    if not s:
        return ""
    s = s.replace(":", "_").replace("-", "_").strip()
    s2 = re.sub(r"[^A-Za-z0-9_ ]+", "", s).upper().strip()
    s2 = re.sub(r"\s+", "_", s2)
    return RELATION_ALIASES.get(s2, s2)


def first_existing(d: dict, keys: list[str]) -> Any:
    for k in keys:
        if k in d and clean_text(d[k]):
            return d[k]
    return None


def collect_node_labels(obj: Any, node_labels: dict[str, str] | None = None) -> dict[str, str]:
    node_labels = node_labels or {}
    if isinstance(obj, dict):
        id_val = first_existing(obj, ["id", "node_id", "candidate_id", "concept_id", "uri", "iri"])
        label_val = first_existing(obj, ["label", "canonical_label", "name", "text"])
        if id_val and label_val:
            node_labels[clean_text(id_val)] = clean_text(label_val)
        for v in obj.values():
            collect_node_labels(v, node_labels)
    elif isinstance(obj, list):
        for item in obj:
            collect_node_labels(item, node_labels)
    return node_labels


def resolve_endpoint(x: Any, node_labels: dict[str, str]) -> str:
    txt = clean_text(x)
    if not txt:
        return ""
    return node_labels.get(txt, txt)


def relation_from_dict(d: dict, node_labels: dict[str, str]) -> dict[str, Any] | None:
    head_raw = first_existing(d, [
        "subject_label", "subject", "head", "source_label", "source", "source_candidate_label", "from", "source_id", "subject_id"
    ])
    rel_raw = first_existing(d, [
        "predicate", "predicate_label", "relation", "relation_label", "type", "edge_label", "property", "label"
    ])
    tail_raw = first_existing(d, [
        "object_label", "object", "tail", "target_label", "target", "target_candidate_label", "to", "target_id", "object_id"
    ])

    head = resolve_endpoint(head_raw, node_labels)
    relation = normalize_predicate(rel_raw)
    tail = resolve_endpoint(tail_raw, node_labels)

    if not (head and relation and tail):
        return None
    if relation.lower() in {"node", "edge", "relation", "triple", "entity", "concept"}:
        return None

    return {
        "subject_label": head,
        "predicate": relation,
        "object_label": tail,
        "confidence": d.get("confidence"),
        "evidence": clean_text(d.get("evidence") or d.get("justification") or d.get("source_text")),
        "raw": d,
    }


def extract_triples_recursive(obj: Any, node_labels: dict[str, str] | None = None) -> list[dict[str, Any]]:
    node_labels = node_labels or collect_node_labels(obj)
    triples = []
    if isinstance(obj, dict):
        r = relation_from_dict(obj, node_labels)
        if r:
            triples.append(r)
        for key, value in obj.items():
            # Avoid re-parsing huge raw strings as dictionaries.
            if isinstance(value, (dict, list)):
                triples.extend(extract_triples_recursive(value, node_labels))
    elif isinstance(obj, list):
        for item in obj:
            triples.extend(extract_triples_recursive(item, node_labels))
    return dedup_triples(triples)


def dedup_triples(triples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    seen = set()
    out = []
    for t in triples:
        s = clean_text(t.get("subject_label") or t.get("head") or t.get("subject"))
        p = normalize_predicate(t.get("predicate") or t.get("predicate_label") or t.get("relation"))
        o = clean_text(t.get("object_label") or t.get("tail") or t.get("object"))
        if not (s and p and o):
            continue
        key = (s.lower(), p.upper(), o.lower())
        if key in seen:
            continue
        seen.add(key)
        out.append({
            "triple_id": t.get("triple_id") or f"pred_{len(out):05d}",
            "subject_label": s,
            "predicate": p,
            "object_label": o,
            "confidence": t.get("confidence"),
            "evidence": t.get("evidence") or "",
            "raw": t.get("raw") or t,
        })
    return out


def load_triples_file(path: Path) -> list[dict[str, Any]]:
    path = Path(path)
    if not path.exists():
        return []
    text = path.read_text(encoding="utf-8", errors="replace").strip()
    if not text:
        return []

    # JSONL support.
    if path.suffix.lower() == ".jsonl" or "\n" in text and text.lstrip().startswith("{"):
        jsonl_triples = []
        bad = 0
        for line in text.splitlines():
            line = line.strip().strip(",")
            if not line or line.startswith("```"):
                continue
            try:
                obj = json.loads(line)
                jsonl_triples.extend(extract_triples_recursive(obj))
            except Exception:
                bad += 1
        if jsonl_triples:
            if bad:
                print(f"[WARN] {path.name}: parsed JSONL with {bad} bad lines ignored")
            return dedup_triples(jsonl_triples)

    obj = json.loads(text)
    return extract_triples_recursive(obj)


def artifact_from_triples(triples: list[dict[str, Any]], *, run_id: str, method: str, source_path: str | None = None) -> EvaluationArtifact:
    doc_id = "xquality"
    artifact = EvaluationArtifact(
        method=method,
        dataset="xquality",
        profile=PROFILE_NAME,
        run_id=run_id,
        metadata={"source_path": source_path or ""},
    )
    artifact.documents.append(EvalDocument(document_id=doc_id, metadata={"dataset": "xquality"}, source_path=source_path))

    entities = {}
    relations = []

    for i, t in enumerate(triples):
        head = clean_text(t.get("subject_label"))
        rel = normalize_predicate(t.get("predicate"))
        tail = clean_text(t.get("object_label"))
        if not (head and rel and tail):
            continue
        entities[head.lower()] = EvalEntity(label=head)
        entities[tail.lower()] = EvalEntity(label=tail)
        relations.append(EvalRelation(
            head=head,
            relation=rel,
            tail=tail,
            evidence=clean_text(t.get("evidence")),
            confidence=t.get("confidence") if isinstance(t.get("confidence"), (int, float)) else None,
            provenance_present=bool(t.get("chunk_id") or t.get("evidence")),
            provenance={"chunk_id": clean_text(t.get("chunk_id"))} if t.get("chunk_id") else {},
            raw=t.get("raw") if isinstance(t.get("raw"), dict) else t,
        ))

    artifact.entities_by_doc[doc_id] = sorted(entities.values(), key=lambda e: e.label.lower())
    artifact.relations_by_doc[doc_id] = relations
    return artifact


def flatten_metric_row(extraction: dict, *, series: str, layer_name: str, stop_index: int | None, evaluation_mode: str, triple_count: int, source_folder: Path | str) -> dict[str, Any]:
    ent = extraction.get("entity", {})
    rel = extraction.get("relation", {})
    counts = extraction.get("counts", {})
    return {
        "series": series,
        "layer_name": layer_name,
        "stop_index": stop_index,
        "evaluation_mode": evaluation_mode,
        "profile": PROFILE_NAME,
        "triple_count": triple_count,
        "source_folder": str(source_folder),
        "gold_entities": counts.get("gold_entities"),
        "gold_relations": counts.get("gold_relations"),
        "pred_entities": counts.get("pred_entities"),
        "pred_relations": counts.get("pred_relations"),
        "entity_tp": ent.get("tp"),
        "entity_fp": ent.get("fp"),
        "entity_fn": ent.get("fn"),
        "entity_precision": ent.get("precision"),
        "entity_recall": ent.get("recall"),
        "entity_f1": ent.get("f1"),
        "relation_tp": rel.get("tp"),
        "relation_fp": rel.get("fp"),
        "relation_fn": rel.get("fn"),
        "relation_precision": rel.get("precision"),
        "relation_recall": rel.get("recall"),
        "relation_f1": rel.get("f1"),
    }


def assert_gold_counts_in_extraction(extraction: dict, label: str):
    counts = extraction.get("counts", {})
    if counts.get("gold_relations", 0) == 0 or counts.get("gold_entities", 0) == 0:
        raise RuntimeError(
            f"Invalid evaluation for {label}: gold counts are zero: {counts}. "
            "This means the wrong gold file/schema was loaded."
        )


## 5. Evaluate completed prefix stop-after-layer outputs

In [ ]:
def is_completed_output_folder(folder: Path) -> bool:
    if not folder.is_dir():
        return False
    triples_path = folder / "triples.json"
    if not triples_path.exists() or triples_path.stat().st_size == 0:
        return False
    # If COMPLETED.json exists, trust it. If not, still evaluate because old notebooks did not always write it.
    return True

prefix_folders = sorted([p for p in PREFIX_OUTPUT_DIR.glob("prefix_stop_after_*") if is_completed_output_folder(p)])
print("Completed prefix folders found:", len(prefix_folders))
for p in prefix_folders:
    print(" -", p.name)

rows = []
per_relation_rows = []
raw_results = {}

for folder in tqdm(prefix_folders, desc="Evaluating prefix outputs"):
    triples_path = folder / "triples.json"
    triples = load_triples_file(triples_path)

    artifact = artifact_from_triples(
        triples,
        run_id=folder.name,
        method="prefix_stop_after_layer_generated_output",
        source_path=str(triples_path),
    )
    extraction = evaluate_extraction(artifact, gold_artifact, profile)
    assert_gold_counts_in_extraction(extraction, folder.name)

    m = re.match(r"prefix_stop_after_(\d+)_", folder.name)
    stop_index = int(m.group(1)) if m else None

    rows.append(flatten_metric_row(
        extraction,
        series="prefix_stop_after_layer_generated_output",
        layer_name=folder.name,
        stop_index=stop_index,
        evaluation_mode="exact_neoolaf_evaluate_extraction",
        triple_count=len(triples),
        source_folder=folder,
    ))

    for pr in extraction.get("per_relation", []):
        pr2 = dict(pr)
        pr2.update({
            "series": "prefix_stop_after_layer_generated_output",
            "layer_name": folder.name,
            "stop_index": stop_index,
            "profile": PROFILE_NAME,
        })
        per_relation_rows.append(pr2)

    raw_results[folder.name] = extraction

summary_df = pd.DataFrame(rows).sort_values(["stop_index", "layer_name"], na_position="last")
per_relation_df = pd.DataFrame(per_relation_rows)

print("Summary rows:", len(summary_df))
display(summary_df)


## 6. Add native NeoOLAF Layer 12 state reference

This uses the original `evaluate_layer_state(...)` helper directly.

In [ ]:
def find_layer12_states() -> list[Path]:
    run_root = ROOT / "examples" / "XQualityMachine32" / "runs"
    candidates = list(run_root.rglob("layer12_serialization/state.json"))
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates

LAYER12_STATE_CANDIDATES = find_layer12_states()
print("Layer 12 state candidates:", len(LAYER12_STATE_CANDIDATES))
for p in LAYER12_STATE_CANDIDATES[:10]:
    print(" -", p)

ADD_NATIVE_LAYER12_STATE_REFERENCE = True
NATIVE_LAYER12_STATE = LAYER12_STATE_CANDIDATES[0] if LAYER12_STATE_CANDIDATES else None
print("Selected NATIVE_LAYER12_STATE:", NATIVE_LAYER12_STATE)

if ADD_NATIVE_LAYER12_STATE_REFERENCE and NATIVE_LAYER12_STATE is not None:
    native_state_result_path = EVAL_OUTPUT_DIR / "native_layer12_state_eval.json"
    native_result = evaluate_layer_state(
        state_path=NATIVE_LAYER12_STATE,
        gold_path=GOLD_PATH,
        profile_name=PROFILE_NAME,
        layer_name="native_neoolaf_layer12_state",
        output_path=native_state_result_path,
    )
    extraction = native_result["extraction"]
    assert_gold_counts_in_extraction(extraction, "native_neoolaf_layer12_state")

    rows.append(flatten_metric_row(
        extraction,
        series="native_neoolaf_layer12_state",
        layer_name="native_neoolaf_layer12_state",
        stop_index=12,
        evaluation_mode="evaluate_layer_state",
        triple_count=extraction.get("counts", {}).get("pred_relations"),
        source_folder=NATIVE_LAYER12_STATE.parent,
    ))

    for pr in extraction.get("per_relation", []):
        pr2 = dict(pr)
        pr2.update({
            "series": "native_neoolaf_layer12_state",
            "layer_name": "native_neoolaf_layer12_state",
            "stop_index": 12,
            "profile": PROFILE_NAME,
        })
        per_relation_rows.append(pr2)

summary_df = pd.DataFrame(rows).sort_values(["series", "stop_index", "layer_name"], na_position="last")
per_relation_df = pd.DataFrame(per_relation_rows)
display(summary_df)


## 7. Add native final export JSON reference, when available

This is useful because your known final export evaluation is the reference result with approximately relation F1 = 0.6108. The notebook tries to find `kg_local.json`, `kg_inferred.json`, `triples.json`, or `kg.json` near the selected Layer 12 run.

In [ ]:
def candidate_final_export_jsons() -> list[Path]:
    candidates = []
    search_roots = []
    if NATIVE_LAYER12_STATE is not None:
        search_roots.extend([NATIVE_LAYER12_STATE.parent, NATIVE_LAYER12_STATE.parent.parent, NATIVE_LAYER12_STATE.parent.parent.parent])
    search_roots.append(ROOT / "examples" / "XQualityMachine32" / "runs")

    preferred_names = ["triples.json", "kg_local.json", "kg_inferred.json", "kg.json", "final_kg.json"]
    for sr in search_roots:
        if not sr.exists():
            continue
        for name in preferred_names:
            candidates.extend(sr.rglob(name))

    # De-duplicate and sort: prefer files near selected layer12 and recently modified.
    seen = set()
    out = []
    for p in candidates:
        if p in seen:
            continue
        seen.add(p)
        if "prefix_stop_after" in str(p):
            continue
        out.append(p)
    out.sort(key=lambda p: (0 if NATIVE_LAYER12_STATE and str(NATIVE_LAYER12_STATE.parent.parent) in str(p) else 1, -p.stat().st_mtime))
    return out

ADD_NATIVE_FINAL_EXPORT_REFERENCE = True
export_candidates = candidate_final_export_jsons()
print("Final export JSON candidates:", len(export_candidates))
for p in export_candidates[:20]:
    try:
        triples = load_triples_file(p)
        print(f" - {p} | triples={len(triples)}")
    except Exception as e:
        print(f" - {p} | ERROR {e}")

# Pick the first candidate with at least one extractable triple.
NATIVE_FINAL_EXPORT_JSON = None
if export_candidates:
    for p in export_candidates:
        try:
            if len(load_triples_file(p)) > 0:
                NATIVE_FINAL_EXPORT_JSON = p
                break
        except Exception:
            pass

print("Selected NATIVE_FINAL_EXPORT_JSON:", NATIVE_FINAL_EXPORT_JSON)

if ADD_NATIVE_FINAL_EXPORT_REFERENCE and NATIVE_FINAL_EXPORT_JSON is not None:
    triples = load_triples_file(NATIVE_FINAL_EXPORT_JSON)
    artifact = artifact_from_triples(
        triples,
        run_id="native_neoolaf_final_export",
        method="native_neoolaf_final_export",
        source_path=str(NATIVE_FINAL_EXPORT_JSON),
    )
    extraction = evaluate_extraction(artifact, gold_artifact, profile)
    assert_gold_counts_in_extraction(extraction, "native_neoolaf_final_export")

    rows.append(flatten_metric_row(
        extraction,
        series="native_neoolaf_final_export",
        layer_name="native_neoolaf_final_export",
        stop_index=12,
        evaluation_mode="exact_neoolaf_evaluate_extraction_from_export_json",
        triple_count=len(triples),
        source_folder=NATIVE_FINAL_EXPORT_JSON,
    ))

    for pr in extraction.get("per_relation", []):
        pr2 = dict(pr)
        pr2.update({
            "series": "native_neoolaf_final_export",
            "layer_name": "native_neoolaf_final_export",
            "stop_index": 12,
            "profile": PROFILE_NAME,
        })
        per_relation_rows.append(pr2)

summary_df = pd.DataFrame(rows).sort_values(["series", "stop_index", "layer_name"], na_position="last")
per_relation_df = pd.DataFrame(per_relation_rows)
display(summary_df)


## 8. Sanity checks

If gold is loaded correctly, every row must have non-zero `gold_entities` and `gold_relations`.

In [ ]:
if summary_df.empty:
    raise RuntimeError("No evaluation rows were produced.")

bad_gold_rows = summary_df[(summary_df["gold_entities"].fillna(0) == 0) | (summary_df["gold_relations"].fillna(0) == 0)]
if not bad_gold_rows.empty:
    display(bad_gold_rows)
    raise RuntimeError("Some rows have zero gold counts. Evaluation is invalid and must not be used.")

print("Gold counts are valid for all rows.")
print("Gold entities:", summary_df["gold_entities"].dropna().unique())
print("Gold relations:", summary_df["gold_relations"].dropna().unique())

display(summary_df[[
    "series", "layer_name", "stop_index", "profile", "triple_count",
    "gold_relations", "relation_tp", "relation_fp", "relation_fn",
    "relation_precision", "relation_recall", "relation_f1",
    "entity_f1"
]])


## 9. Save outputs

In [ ]:
summary_path = EVAL_OUTPUT_DIR / "summary_exact_eval_fixed.csv"
per_relation_path = EVAL_OUTPUT_DIR / "per_relation_exact_eval_fixed.csv"
raw_path = EVAL_OUTPUT_DIR / "raw_extraction_results_fixed.json"

def json_default(o):
    try:
        return asdict(o)
    except Exception:
        return str(o)

summary_df.to_csv(summary_path, index=False)
per_relation_df.to_csv(per_relation_path, index=False)
raw_path.write_text(json.dumps(raw_results, indent=2, ensure_ascii=False, default=json_default), encoding="utf-8")

print("Saved:")
print(" -", summary_path)
print(" -", per_relation_path)
print(" -", raw_path)


## 10. Plots

In [ ]:
plot_df = summary_df.copy()
plot_df["x_label"] = plot_df.apply(lambda r: f"{int(r['stop_index'])}" if pd.notna(r.get("stop_index")) else str(r["layer_name"]), axis=1)

# Relation F1 plot
plt.figure(figsize=(12, 5))
for series, sdf in plot_df.groupby("series"):
    sdf = sdf.sort_values("stop_index")
    plt.plot(sdf["x_label"], sdf["relation_f1"], marker="o", label=series)
plt.xlabel("Stop point / reference")
plt.ylabel("Relation F1")
plt.title("Exact NeoOLAF evaluation: prefix stop-after-layer outputs vs native references")
plt.xticks(rotation=45, ha="right")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
fig_path = EVAL_OUTPUT_DIR / "relation_f1_exact_eval_fixed.png"
plt.savefig(fig_path, dpi=200)
plt.show()
print("Saved plot:", fig_path)

# Relation precision/recall/F1 table display
cols = ["series", "layer_name", "stop_index", "triple_count", "relation_precision", "relation_recall", "relation_f1", "entity_f1"]
display(summary_df[cols].sort_values(["series", "stop_index"], na_position="last"))
